# Serving LLMs Efficiently with vLLM - Part II

In this notebook, you'll learn how to:
1. 启动一个 serving 真实模型的 vLLM 推理服务器。
2. 用OpenAI兼容API查询它
3. 用logprobs和采样参数探索模型行为
4. 通过实时Prometheus指标观察continuous batching和KV Cache使用
5. 演示前缀缓存

是一条完整的在线Serving证据链：启动服务 → 发请求 → 控行为 → 看调度/显存 → 省计算  




## From Quantization to Serving

在之前的课程中，你用llm-compressor把模型权重从16-bit用量化到4-bit。现在你要用vLLM来serve这个模型，并通过OpenAI兼容API与它交互。

[vLLM](https://github.com/vllm-project/vllm) is an open-source LLM inference engine that integrates key serving optimizations:

| Feature | Benefit |
|:--|:--|
| **Continuous batching** | token级调度：不用等最长的请求，算力不空转 |
| **PagedAttention** | KV Cache按定长block管理：内存浪费接近零 |
| **Prefix caching** | 跨请求复用共享prompt前缀的KV Cache |
| **Quantization support** | 原生serve GPTQ/AWQ/compressed-tensors模型 |
| **OpenAI-compatible API** | 已用OpenAI client的应用可直接替换 |

## Step 1: Start Your vLLM Server

In this learning environment, a pre-warmed vLLM server is provided to you to use with Qwen3-0.6B, so the server is already running and you'll use the code cells in this notebook to interact with it.

**How to serve a model with vLLM?**

To serve your model with vLLM, you'd need to run this command in the terminal. 

```bash
vllm serve Qwen/Qwen3-0.6B --dtype=bfloat16 --max-model-len 4096
```

You can find the installation details [here](https://docs.vllm.ai/en/latest/getting_started/installation/).

- **`vllm serve`**: launches vLLM's built-in inference server. It loads the model weights into the engine (with PagedAttention, continuous batching, and prefix caching enabled by default) and exposes it over HTTP on port 8000.
- **`Qwen/Qwen3-0.6B`**: the model identifier on the [Hugging Face Hub](https://huggingface.co/Qwen/Qwen3-0.6B). On first run, vLLM downloads the weights, tokenizer, and config from Hugging Face into the local cache (`~/.cache/huggingface/hub`), then loads them into memory. Subsequent runs reuse the cached files.
- **`--dtype=bfloat16`**: loads the weights in BF16 precision. 
- **`--max-model-len 4096`**: caps the context window (prompt + generation) at 4096 tokens. 

> **Note:** You can also serve the quantized model from the previous lesson, but that requires a GPU because its W4A16 format only has optimized runtime support on GPUs. Since this learning environment runs on CPU, the original model is the one served through vLLM.

vLLM包了一层OpenAI兼容HTTP API，实现了SDK调的同样路由（/v1/models等），请求/响应形状相同。 (`/v1/models`, `/v1/chat/completions`, `/v1/completions`, `/v1/embeddings`) with the same request and response shapes.

So here let's check if it's running. 

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, math, sys

VLLM_URL = "http://localhost:8000"
os.makedirs("outputs", exist_ok=True)

print("Waiting for vLLM server...")
for attempt in range(60):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            MODEL = r.json()["data"][0]["id"]
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
    if attempt % 6 == 5:
        print(f"  Still waiting... ({(attempt + 1) * 5}s elapsed)")
else:
    raise RuntimeError(
        "vLLM server not reachable after 5 minutes."
    )

print(f"Connected to {VLLM_URL} — model: {MODEL}")

Waiting for vLLM server...
Connected to http://localhost:8000 — model: Qwen/Qwen3-0.6B


> **Note:** The vLLM server might need 1 or 2 minutes to be ready.

## Your First Local LLM Request

vLLM 暴露了与 OpenAI 兼容的 API，因此我们使用标准的 openai Python 客户端。

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")


# 2. base_url=f"{VLLM_URL}/v1"：覆盖默认的 https://api.openai.com/v1，
# 改指你的服务，比如 http://localhost:8000/v1。
# 必须带 /v1，否则路径对不上 vLLM 的 api_router.py。
# - VLLM_URL 是你前面定义的变量（推测），f"..." 只是字符串插值。

# 3. api_key="unused"：OpenAI 类强制要求传 api_key（会填进 Authorization: Bearer ... 头），
# 但本地 vLLM 默认不鉴权，所以填任意占位字符串即可。
# 这是功能正确 vs 性能最优无关的纯接口兼容手段。



Qwen3 支持一种 thinking mode（思考模式），会在回答前先生成 chain-of-thought（思维链）推理。我们用 enable_thinking: False 把它关掉，让回复更短、更可预测。

In [3]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20, 
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {resp.usage.total_tokens} total")

Response (2.02s, 36 tokens):
Paged Attention is a technique that allows for efficient attention to a sequence of tokens by dividing the attention space into smaller blocks, each corresponding to a specific position in the input.

Usage: 21 prompt + 36 completion = 57 total


## Exploring Model Behavior

除了简单聊天，vLLM API 还让你能查看模型内部状态并控制生成。例如，Logprobs： 让你看到模型对生成的每个 token 的确信度，以及它考虑过的备选项。

In [4]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=15,
    temperature=0.0,#保证可复现
    logprobs=True,#开关。让 Server 把每步已算好的 log_softmax(logits) 回传，不改变生成，只加一点 CPU 拷贝。
    top_logprobs=5,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(f"Response: {resp.choices[0].message.content.strip()}\n")
print("Token-by-token probabilities:\n")

for tok in resp.choices[0].logprobs.content[:8]:
    print(f"  Chosen: '{tok.token}'  (logprob {tok.logprob:.2f})")
    if tok.top_logprobs:
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "\u2588" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  '{alt.token}'")
    print()

Response: The capital of France is **Paris**.

Token-by-token probabilities:

  Chosen: 'The'  (logprob -0.00)
     99.9%  ███████████████████  'The'
      0.1%  █  'France'
      0.0%  █  'Capital'
      0.0%  █  'Paris'
      0.0%  █  'Le'

  Chosen: ' capital'  (logprob -0.00)
    100.0%  ███████████████████  ' capital'
      0.0%  █  ' Capital'
      0.0%  █  ' **'
      0.0%  █  ' French'
      0.0%  █  ' official'

  Chosen: ' of'  (logprob -0.00)
     99.9%  ███████████████████  ' of'
      0.0%  █  ' city'
      0.0%  █  ' is'
      0.0%  █  ' ('
      0.0%  █  ' and'

  Chosen: ' France'  (logprob -0.00)
    100.0%  ███████████████████  ' France'
      0.0%  █  ' **'
      0.0%  █  ' the'
      0.0%  █  'France'
      0.0%  █  ' Europe'

  Chosen: ' is'  (logprob -0.00)
    100.0%  ███████████████████  ' is'
      0.0%  █  ' in'
      0.0%  █  ','
      0.0%  █  ' was'
      0.0%  █  ' **'

  Chosen: ' **'  (logprob -0.08)
     92.2%  ██████████████████  ' **'
      6.7%  █  '

## Observing vLLM Under the Hood

vLLM 暴露了一个 Prometheus 兼容的 /metrics 端点（一种很容易被抓取的格式）。要盯的关键指标：
- num_requests_running / waiting：正在执行 vs 排队等的请求数
- gpu_cache_usage_perc（或cpu_cache_usage_perc）：KV Cache 内存压力
- prompt_tokens_total / generation_tokens_total：累计的 prompt / 生成 token 数



In [5]:
def get_vllm_metrics(base_url=VLLM_URL):
    """Scrape vLLM Prometheus /metrics and return {name: value}."""
    r = requests.get(f"{base_url}/metrics")
    metrics = {}
    for line in r.text.split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except (ValueError, IndexError):
            continue
    return metrics

metrics = get_vllm_metrics()
print("Current vLLM Metrics:")
for key in ["vllm:num_requests_running", "vllm:num_requests_waiting",
            "vllm:gpu_cache_usage_perc", "vllm:cpu_cache_usage_perc",
            "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]:
    if key in metrics:
        print(f"  {key.replace('vllm:', '')}: {metrics[key]:g}")

with open("outputs/metrics_snapshot.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nFull metrics saved to outputs/metrics_snapshot.json")

Current vLLM Metrics:
  num_requests_running: 0
  num_requests_waiting: 0
  prompt_tokens_total: 38
  generation_tokens_total: 46

Full metrics saved to outputs/metrics_snapshot.json


## Continuous Batching in Action

vLLM 使用连续批处理（迭代级调度）：当一个请求生成结束，其空位立刻被下一个等待请求填上。
我们并发发5个请求，并在它们运行时观察指标。

In [ ]:
import concurrent.futures

prompts = [
    "What is quantization?",
    "Explain KV caching briefly.",
    "What is continuous batching?",
    "Why is LLM inference memory-bound?",
    "What is PagedAttention?",
]

def _ask(prompt):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

before = get_vllm_metrics()
print(f"Sending {len(prompts)} concurrent requests...\n")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(prompts)) as pool:
    futures = {pool.submit(_ask, p): p for p in prompts}
    time.sleep(0.5)
    during = get_vllm_metrics()
    running = during.get("vllm:num_requests_running", "--")
    waiting = during.get("vllm:num_requests_waiting", "--")
    print(f"  [mid-flight]  running: {running}  |  waiting: {waiting}")

    for f in concurrent.futures.as_completed(futures):
        resp = f.result()
        print(f"  done: \"{futures[f][:40]}\" -> {resp.usage.completion_tokens} tokens")


elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0)

print(f"\nAll {len(prompts)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

Sending 5 concurrent requests...

  [mid-flight]  running: 5.0  |  waiting: 0.0
  done: "What is quantization?" -> 60 tokens
  done: "Explain KV caching briefly." -> 60 tokens
  done: "Why is LLM inference memory-bound?" -> 60 tokens
  done: "What is PagedAttention?" -> 60 tokens
  done: "What is continuous batching?" -> 60 tokens

All 5 completed in 0.82s
Tokens generated: 300  |  ~364.4 tokens/s


>**注意**：第一次跑这格，你可能看到running:0或1而不是5。再跑一次就会显示5。指标只在发出请求后一瞬间采样一次，读得太早，请求可能还没到服务端，所以显示0（既没在跑也没排队），尽管5个马上就要一起跑了。重复采样才能看到真正的峰值5。

**What Just Happened**

vLLM的调度器收到全部5个请求，用连续批处理处理。每步生成为batch里每个请求各产token，做完的立刻让出槽位。  
vLLM的PagedAttention是规模化的关键：KV Cache被切成定长block，可放在内存任意位置。请求结束时它的block立刻可复用——无碎片。

## Prefix Caching

很多应用在请求间共享很长的system prompt。没有前缀缓存时，vLLM每次都要为共享前缀重算KV Cache。
开了前缀缓存，vLLM能识别共享前缀并复用已缓存的KV。第一个请求付全量prefill代价，后续请求跳过它。  
我们发5个带相同system prompt的请求，对比耗时。

In [8]:
SYSTEM_PROMPT = (
    "You are a helpful AI teaching assistant for a course on "
    "LLM optimization. You specialize in explaining concepts like "
    "quantization, inference optimization, and model serving. Keep "
    "answers concise -- one or two sentences."
)

questions = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]

before = get_vllm_metrics()
timings = []
tok_counts = []

print("Sending 5 requests with the SAME system prompt...\n")
for i, q in enumerate(questions):
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    dt = time.time() - t0
    timings.append(dt)
    tok_counts.append(resp.usage.completion_tokens)
    tokens = resp.usage.completion_tokens
    ms_per_tok = (dt / tokens * 1000) if tokens > 0 else 0.0
    print(f"  [{i+1}] {q:<35} {dt:.2f}s  ({tokens} tok, {ms_per_tok:.0f} ms/tok)")
    
after = get_vllm_metrics()

prefix_before = before.get("vllm:prefix_cache_queries_total", 0)
prefix_after = after.get("vllm:prefix_cache_queries_total", 0)

print(f"\nPrefix cache queries: {prefix_before:g} -> {prefix_after:g}  (+{prefix_after - prefix_before:g})")

cache_keys = [k for k in after if "prefix" in k.lower() 
              or "cache_hit" in k.lower()]
for k in sorted(cache_keys):
    b, a = before.get(k, 0), after.get(k, 0)
    if a != b and k != "vllm:prefix_cache_queries_total":
        print(f"  {k}: {b:g} -> {a:g}")

print("\n The increasing prefix_cache_queries count confirms vLLM is ")
print("checking and reusing cached KV blocks for the shared system prompt.")

Sending 5 requests with the SAME system prompt...

  [1] What is weight quantization?        0.66s  (37 tok, 18 ms/tok)
  [2] How does vLLM handle memory?        0.68s  (39 tok, 17 ms/tok)
  [3] What is continuous batching?        0.52s  (33 tok, 16 ms/tok)
  [4] Why use prefix caching?             0.36s  (25 tok, 14 ms/tok)
  [5] What is GPTQ?                       0.75s  (44 tok, 17 ms/tok)

Prefix cache queries: 443 -> 758  (+315)
  vllm:prefix_cache_hits_total: 192 -> 448

 The increasing prefix_cache_queries count confirms vLLM is 
checking and reusing cached KV blocks for the shared system prompt.


**Why Prefix Caching Matters**

prefix_cache_queries增长证明vLLM正在为共享system prompt复用已缓存的KV。你的短system prompt（~50 token）省的量相对总生成时间很小。但生产中长system prompt（几千token的指令或few-shot样例），前缀缓存能省掉每次请求的一大块prefill计算。

---

## (Optional) KV Cache Size Visualization for Qwen3-0.6B

The KV cache stores key and value tensors from the attention mechanism. It grows **linearly** with sequence length and concurrent requests.

In [ ]:
num_layers = 28
num_kv_heads = 8     # GQA: 16 Q heads, 8 KV heads
head_dim = 128
dtype_bytes = 2      # BF16

per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes

print(f"KV Cache -- Qwen3-0.6B")
print(f"Per token: 2 x {num_layers} x {num_kv_heads} x {head_dim} x {dtype_bytes}"
      f" = {per_token:,} bytes ({per_token // 1024} KB)\n")
print(f"  {'Context':>8}  {'KV Cache':>10}")
print(f"  {'-'*8}  {'-'*10}")
for ctx in [1, 64, 256, 1024, 4096]:
    size = per_token * ctx
    label = f"{size / 1024:.0f} KB" if size < 1024**2 else f"{size / 1024**2:.0f} MB"
    print(f"  {ctx:>7}t  {label:>10}")

print(f"\n  10 concurrent x 4096 ctx = {per_token * 4096 * 10 / 1024**3:.1f} GB")

---

## (Optional) Thinking Mode

Qwen3支持思考模式，模型先在<think>...</think>里生成内部思维链，再给可见答案。答得更好，但用多得多的token——更多KV Cache、更多计算、更长耗时。  
我们在同一prompt上并排流式对比两种模式。

In [13]:
prompt = "What makes continuous batching better than static batching?中文"

for label, thinking, max_tok in [
    ("Thinking OFF", False, 80), ("Thinking ON", True, 2000)]:
    print(f"=== {label} ===\n")
    start = time.time()
    tokens = 0
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok, temperature=0.7, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            sys.stdout.write(chunk.choices[0].delta.content)
            sys.stdout.flush()
            tokens += 1
    elapsed = time.time() - start
    print(f"\n  [{tokens} tokens, {elapsed:.2f}s]\n")

=== Thinking OFF ===

连续批次（Continuous Batching）相比静态批次（Static Batching）的优势主要体现在以下几个方面：

1. **提高效率**  
   连续批次能够实时调整批次的生产参数，减少等待时间，提高整体生产效率。

2. **优化资源利用率**  
   通过动态调整，可以更有效地利用生产资源，减少浪费。

3. **适应性强**
  [79 tokens, 0.69s]

=== Thinking ON ===

<think>
嗯，用户问的是为什么连续批次比静态批次更好。首先，我需要理解这两个概念的区别。静态 batching 是指将一批物料一次性放入，然后进行处理，通常是在一个固定的批次中完成。而连续 batching 则是在不同的时间点或不同的批次之间进行，可能更灵活。

接下来，我应该考虑连续批次的优势。比如，连续批次可以更灵活地调整处理时间，适应不同需求。例如，如果需要快速调整生产计划，连续批次可能更容易。另外，连续批次可以减少库存积压，因为不需要等待整个批次完成才能进行后续处理。

然后，可能需要举例说明，比如连续批次可以在生产开始前调整，或者根据实时数据调整参数，这样能提高效率。另外，连续批次可能更节省资源，因为每次处理不需要额外的资源，比如设备或时间。

还要考虑用户可能的背景。用户可能是生产经理、IT人员或者供应链管理专家，他们可能关心生产流程的优化和效率提升。所以需要从效率、灵活性、成本和资源利用等方面解释。

需要确保回答清晰，结构分明，可能分点列出优势，但用户可能需要更简洁的中文解释。同时，避免使用过于专业的术语，保持口语化，但又要准确。

最后，检查是否有遗漏的关键点，比如连续批次是否能适应不同批次的需求，或者是否能更灵活地进行调整。确保回答全面，同时满足用户的问题需求。
</think>

连续批次（Continuous Batch Processing, CBP）相比静态批次（Static Batch Processing, SBP）的优势主要体现在以下几个方面：

1. **灵活性与可调节性**  
   连续批次允许在不同时间或不同批次中动态调整参数（如处理时间、原料配比等），适应生产计划的实时变化。例如，可以在生产开始前调整工艺参数，或根据市场需求调整批次规模。



## Summary

在这个 Notebook 中，你完成了以下内容：

* 启动了一个**vLLM 服务器**，提供 Qwen3-0.6B 模型服务，并使用 **OpenAI Client** 与服务器建立连接。

* 使用 **logprobs（token 概率）**探索模型内部的生成过程。

* 使用 **`/metrics` 接口**观察 **KV Cache 的使用情况**以及**请求数量**等服务端指标。

* 演示了**连续批处理（Continuous Batching）**：通过并发发送多个请求，观察 vLLM 如何对请求进行动态调度和批处理。

* 演示了**前缀缓存（Prefix Caching）**：当多个请求共享相同的系统提示词（System Prompt）时，可以复用已经缓存的 KV 条目，从而减少重复计算。

* 以**流式方式（Streaming）**运行 **Thinking Mode（思考模式）**，实时观察模型的链式思考过程。


## Resources

- [vLLM Github](https://github.com/vllm-project/vllm)
- [vLLM Docs](https://docs.vllm.ai/en/latest/)
- [vLLM Installation Instructions](https://docs.vllm.ai/en/latest/getting_started/installation/)